# 05 · Seismicity analysis: completeness, b-values and structure

Catalogue statistics need many events. This notebook uses the **reference catalogue** if it
holds enough events with magnitudes (e.g. the full 2012–2014 ARGOS catalogue). Otherwise it
uses **our ML catalogue** from notebook 03. With one day of data, treat the numbers as a
demonstration of the methods, not as results. We will:

1. estimate the magnitude of completeness $M_c$ with two methods and bootstrap uncertainties,
2. estimate the Gutenberg–Richter b-value by maximum likelihood,
3. look at the seismicity in space and time,
4. find spatial clusters with **DBSCAN** and fit planes through them with **PCA**, as a
   data-driven way to find faults and fluid pathways.

All the methods are short functions in `seismoml/stats.py`, written from the original papers.
Read them!

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import seismoml as sml
from seismoml import catalogue, data, plotting, stats

plotting.use_style()
cfg = sml.load_config()

ref = catalogue.read_catalogue(cfg.path("reference_catalogue"))
ref = ref[(ref["latitude"].between(*cfg.lat_range)) & (ref["longitude"].between(*cfg.lon_range))]
ref = catalogue.to_utm(ref, cfg.utm_epsg).reset_index(drop=True)
ml_day = catalogue.to_utm(
    catalogue.read_catalogue(cfg.outputs / "catalogues" / "PN-volpick_events.csv"), cfg.utm_epsg)
stations = data.active_stations(data.station_table(cfg.path("stations")), cfg.day)
print(f"Reference: {len(ref)} events in the study region, {ref['magnitude'].notna().sum()} with magnitude")
print(f"PN-volpick ({cfg.day.date}): {len(ml_day)} events, {ml_day['magnitude'].notna().sum()} with magnitude")

MIN_EVENTS = 200   # events with magnitude needed before we prefer the reference catalogue
if ref["magnitude"].notna().sum() >= MIN_EVENTS:
    cat, cat_name = ref, "reference catalogue"
else:
    cat, cat_name = ml_day, f"PN-volpick catalogue of {cfg.day.date}"
print(f"\nStatistics below use the {cat_name} ({len(cat)} events).")
if cat["magnitude"].notna().sum() < MIN_EVENTS:
    print(f"Fewer than {MIN_EVENTS} magnitudes: estimates are illustrative only. "
          "Add a longer catalogue to data/reference/ for robust results.")

## 1 · Activity through time

In [ ]:
span = cat["time"].max() - cat["time"].min() if len(cat) else pd.Timedelta(0)
freq, unit = ("1h", "hour") if span <= pd.Timedelta(days=3) else ("1D", "day")
fig, axes = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True)
axes[0].scatter(cat["time"], cat["magnitude"], s=10, color="k", alpha=0.7)
axes[0].set_ylabel("Magnitude")
counts = cat.set_index("time").resample(freq).size()
axes[1].bar(counts.index, counts.values, width=pd.Timedelta(freq) * 0.9, align="edge", color="#4c72b0")
ax2 = axes[1].twinx()
ax2.plot(cat["time"], np.arange(1, len(cat) + 1), color="#d62728", lw=1.4)
ax2.set_ylabel("Cumulative events", color="#d62728")
ax2.grid(False)
axes[1].set_ylabel(f"Events per {unit}")
axes[0].set_title(f"Activity in the {cat_name}");

In [ ]:
day0 = pd.Timestamp(cfg.day.datetime, tz="UTC")
ref_day = ref[(ref["time"] >= day0) & (ref["time"] < day0 + pd.Timedelta(days=1))]
print(f"Reference events in the study region on {cfg.day.date}: {len(ref_day)}")
print(f"PN-volpick events on {cfg.day.date}: {len(ml_day)}  →  "
      f"{len(ml_day) / max(len(ref_day), 1):.1f}× the reference")

## 2 · Frequency–magnitude distribution and $M_c$

Two estimators of completeness:

* **MAXC** (maximum curvature; Wiemer & Wyss, 2000). $M_c$ is the bin with the most events.
  It is fast but biased low for gradually curved distributions, so +0.2 is often added.
* **MBS** (b-value stability; Cao & Gao, 2002; Woessner & Wiemer, 2005). $M_c$ is the
  smallest cutoff above which $b$ stops changing within its uncertainty.

The b-value uses the Aki (1965) maximum-likelihood estimator with Utsu's bin correction,
and the uncertainty of Shi & Bolt (1982):

$$ b = \frac{\log_{10} e}{\bar M - (M_c - \Delta M/2)}, \qquad
   \sigma_b = 2.3\,b^2 \sqrt{\frac{\sum (M_i - \bar M)^2}{N(N-1)}} $$

In [ ]:
DM = 0.1
mags = cat["magnitude"].dropna().to_numpy()
MBS_MIN = max(10, len(mags) // 10)   # minimum events above a trial Mc for the MBS method
fmd = stats.fmd(mags, DM)
mc_maxc = stats.mc_maxc(mags, DM, correction=0.2)
mc_mbs, mbs_table = stats.mc_mbs(mags, DM, window=0.5, min_events=MBS_MIN) if len(mags) >= 20 \
    else (np.nan, pd.DataFrame(columns=["mc", "b", "sigma_b", "n", "b_avg", "stable"]))
print(f"{len(mags)} magnitudes")
print(f"Mc (MAXC + 0.2) = {mc_maxc:.1f}")
print(f"Mc (MBS)        = {mc_mbs:.1f}" + ("" if np.isfinite(mc_mbs) else "  (not stable: too few events)"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, (name, mc) in zip(axes, (("MAXC + 0.2", mc_maxc), ("MBS", mc_mbs))):
    b, sb, a, n = stats.b_value_mle(mags, mc, DM)
    plotting.fmd_plot(fmd, mc, b, a, ax=ax, label=f"{name}: b = {b:.2f} ± {sb:.2f} (N = {n})")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ok = mbs_table["n"] >= MBS_MIN
ax.errorbar(mbs_table.loc[ok, "mc"], mbs_table.loc[ok, "b"], yerr=mbs_table.loc[ok, "sigma_b"],
            fmt="o", ms=3, color="k", label="b(Mc) ± σ")
ax.plot(mbs_table.loc[ok, "mc"], mbs_table.loc[ok, "b_avg"], color="#d62728", label="b averaged over next 0.5")
if np.isfinite(mc_mbs):
    ax.axvline(mc_mbs, color="#4c72b0", ls="--", label=f"Mc (MBS) = {mc_mbs:.1f}")
ax.set(xlabel="Cut-off magnitude", ylabel="b-value", title="b-value stability")
ax.legend();

### How certain are we? Bootstrap

Resample the catalogue with replacement 500 times and recompute $M_c$ and $b$ each time.

In [ ]:
boot_maxc = stats.bootstrap_mc_b(mags, "maxc", n_boot=500, dm=DM, correction=0.2)
boot_mbs = stats.bootstrap_mc_b(mags, "mbs", n_boot=200, dm=DM, window=0.5, min_events=MBS_MIN)
pd.concat({"MAXC+0.2": boot_maxc.summary(), "MBS": boot_mbs.summary()}).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for name, bt, c in (("MAXC+0.2", boot_maxc, "#4c72b0"), ("MBS", boot_mbs, "#dd8452")):
    axes[0].hist(bt.mc[np.isfinite(bt.mc)], bins=np.arange(-1, 3, DM), alpha=0.6, color=c, label=name)
    axes[1].hist(bt.b[np.isfinite(bt.b)], bins=30, alpha=0.6, color=c, label=name)
axes[0].set(xlabel="Mc", ylabel="Bootstrap samples")
axes[1].set(xlabel="b-value")
axes[1].legend();

A b-value well above 1 is typical of volcanic and geothermal areas: high pore pressure,
strong heterogeneity and high thermal gradients favour many small ruptures. Wilks et al.
(2017) reported b = 1.40 ± 0.14 for Aluto. Yemane et al., using a larger relocated
catalogue, found b = 1.97 ± 0.10 and attributed it to fluids.

**Caveat:** at small magnitudes ML does not scale 1:1 with moment magnitude (ML ∝ 1.5 Mw
below about M 2; Deichmann, 2017). b-values from ML catalogues of small events are therefore
inflated compared with Mw-based values. Compare b-values only within one magnitude
type and range.

## 3 · Where is the seismicity?

In [ ]:
fig = plotting.map_and_sections(stations, {"reference": ref, f"PN-volpick {cfg.day.date}": ml_day},
                                cfg.lat_range, cfg.lon_range, cfg.depth_range,
                                title="Seismicity in the study region")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(cat["depth_km"], bins=np.arange(-2.5, 40.5, 0.5), orientation="horizontal", color="#4c72b0")
ax.axhline(0, color="k", lw=0.8)
ax.text(ax.get_xlim()[1] * 0.6, -0.3, "sea level", va="bottom", fontsize=8)
ax.set(ylim=(40, -2.5), xlabel="Events", ylabel="Depth (km b.s.l.)", title="Depth distribution");

## 4 · Clusters and planes

### DBSCAN

DBSCAN groups points that have at least `min_samples` neighbours within a radius `eps`.
Points in sparse regions are labelled noise (−1). It needs no preset number of clusters and
finds arbitrarily shaped clusters, which suits seismicity. `eps` is taken at the "knee" of
the k-distance curve, where distances jump from cluster spacing to background spacing.

Work in a Cartesian frame: UTM easting/northing (km) and depth (km).

In [ ]:
MIN_SAMPLES = 8                     # DBSCAN core-point size, including the point itself
K = MIN_SAMPLES - 1                 # matching neighbour count for the k-distance curve
sel = cat.copy()
xyz = sel[["easting_km", "northing_km", "depth_km"]].to_numpy()
sel["cluster"] = -1
if len(sel) > 3 * MIN_SAMPLES:
    kd = stats.k_distance(xyz, k=K)
    EPS = float(kd[stats.knee_point(kd)])
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(kd, color="k")
    ax.axhline(EPS, color="#d62728", ls="--", label=f"eps = {EPS:.2f} km (knee)")
    ax.set(xlabel="Events sorted by distance", ylabel=f"Distance to {K}th neighbour (km)",
           title="k-distance plot")
    ax.legend()
    sel["cluster"] = stats.dbscan_labels(xyz, eps_km=EPS, min_samples=MIN_SAMPLES)
else:
    print(f"Only {len(sel)} events: too few for density-based clustering.")
counts = sel["cluster"].value_counts().sort_index()
print(f"{(counts.index >= 0).sum()} clusters; {counts.get(-1, 0)} events labelled noise")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
cmap = plt.get_cmap("tab20")
for cid, g in sel.groupby("cluster"):
    kw = dict(s=8, color="0.6", label="noise") if cid < 0 else dict(s=12, color=cmap(cid % 20), label=f"C{cid}")
    axes[0].scatter(g["easting_km"], g["northing_km"], **kw)
    axes[1].scatter(g["easting_km"], g["depth_km"], **kw)
st_xy = catalogue.to_utm(stations, cfg.utm_epsg)
axes[0].scatter(st_xy["easting_km"], st_xy["northing_km"], marker="^", s=60, color="#ff7f0e", edgecolor="k")
axes[0].set(xlabel="Easting (km)", ylabel="Northing (km)", aspect="equal", title="DBSCAN clusters, map")
axes[1].set(xlabel="Easting (km)", ylabel="Depth (km b.s.l.)", title="E–W section")
axes[1].invert_yaxis()
axes[0].legend(fontsize=7, ncols=2, markerscale=1.5);

### PCA: is a cluster a plane, a line or a blob?

The covariance of a cluster's hypocentres has eigenvalues $\lambda_1 \ge \lambda_2 \ge \lambda_3$.

* **Planar** cluster (fault, sheet intrusion): $\lambda_3 \ll \lambda_2$. The eigenvector of
  $\lambda_3$ is the plane's normal, which gives strike and dip.
* **Linear** cluster (conduit, fracture intersection): $\lambda_2 \ll \lambda_1$.

We report planarity $(\lambda_2-\lambda_3)/\lambda_1$ and linearity
$(\lambda_1-\lambda_2)/\lambda_1$, the eigenvalue shape measures common in point-cloud
analysis (e.g. Demantké et al., 2011).

The Main Ethiopian Rift is cut by NNE–SSW Wonji faults, and Aluto sits on one. Do the
clusters line up with them?

In [ ]:
rows = []
for cid, g in sel[sel["cluster"] >= 0].groupby("cluster"):
    if len(g) < 15:
        continue
    fit = stats.hypocentre_pca(g[["easting_km", "northing_km", "depth_km"]].to_numpy())
    rows.append({"cluster": f"C{cid}", "events": len(g),
                 "centroid depth (km)": fit.centroid[2],
                 "strike (°)": fit.strike, "dip (°)": fit.dip,
                 "planarity": fit.planarity, "linearity": fit.linearity,
                 "σ along axis 1 (km)": np.sqrt(fit.eigenvalues[0])})
pca = pd.DataFrame(rows).set_index("cluster") if rows else pd.DataFrame()
pca.round(2) if len(pca) else print("No cluster with at least 15 events to fit.")

In [ ]:
if len(pca):
    planar = pca[pca["planarity"] > 0.3]
    fig = plt.figure(figsize=(4.5, 4.5))
    ax = fig.add_subplot(projection="polar")
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    for name, r in pca.iterrows():
        th = np.radians([r["strike (°)"], r["strike (°)"] + 180])
        ax.plot(th, [r["planarity"]] * 2, "o-", color="#d62728" if name in planar.index else "0.6")
    ax.set_rlim(0, 1)
    ax.set_title("Cluster strikes (radius = planarity)", pad=18)

## Take-aways

* $M_c$ and $b$ depend on the method. Always report the method and a bootstrap uncertainty.
* ML catalogues lower $M_c$. The real test is whether the FMD of a full ML catalogue
  continues the Gutenberg–Richter line of the manual catalogue to smaller magnitudes. That
  needs the ML pipeline run on many days: try it on a week of data as a follow-up project.
* DBSCAN + PCA is a quick, reproducible way to turn a cloud of hypocentres into testable
  structural hypotheses. The results depend on `eps`, and on location quality.

## Exercises

1. Split the reference catalogue at 2 km b.s.l. (shallow hydrothermal vs. deeper). Are
   the b-values different, given their uncertainties? Is a high b-value restricted to
   the shallow hydrothermal system?
2. Vary `EPS` by ±30 % around the knee value. Which clusters are robust?
3. With a long catalogue: compute the b-value through time in windows of 150 events. Is
   there a change around the workshop day?

**That's the end of the workshop. Thank you!**